# Mumbai & Navi Mumbai Air Quality (2015–2020): ML Pipeline

**Dataset:** Kaggle — [Air Quality Data in India (2015–2020)](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india)

**Goal:** Filter station-level observations to Mumbai and Navi Mumbai using station metadata and latitude/longitude, clean the data, train a Random Forest regression model, evaluate it, and export Power BI-ready CSVs.

## Important dataset caveat
The classic Kaggle dataset contains city-level files and station-level files. The city-level file may list Mumbai but not Navi Mumbai. This notebook therefore prioritizes `station_day.csv` joined to `stations.csv` so that coordinates and station metadata can be used. It does **not** relabel Mumbai records as Navi Mumbai. The notebook prints the station/city coverage and stops with a clear warning if both areas cannot be identified from the downloaded data.

The coordinate boxes below are approximate geographic filters and are configurable. Inspect the selected stations on a map before presenting results. If station metadata already explicitly identifies a city, that label takes priority over coordinate-only classification.

**Model interpretation:** This is an AQI-estimation model using same-observation pollutant readings, not a future AQI forecasting model. Since AQI is calculated from pollutant measurements, strong performance does not necessarily mean the model can predict future air quality.


## 1. Install and import dependencies
Run the install line once if any package is missing.

In [ ]:
# If needed, run once:
# %pip install pandas numpy matplotlib seaborn scikit-learn nbformat

import os
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
OUTPUT_DIR = Path("aqi_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Ready. Output folder:", OUTPUT_DIR.resolve())


## 2. Point to the downloaded Kaggle files
Extract the Kaggle ZIP first. Set `DATA_DIR` to the folder containing `station_day.csv` and `stations.csv`. This notebook uses station-level daily observations to support geographic filtering.

In [ ]:
DATA_DIR = Path(".")  # Change this if the CSVs are in another folder

STATION_DAY_FILE = DATA_DIR / "station_day.csv"
STATIONS_FILE = DATA_DIR / "stations.csv"

for path in [STATION_DAY_FILE, STATIONS_FILE]:
    print(path, "exists:", path.exists())

if not STATION_DAY_FILE.exists() or not STATIONS_FILE.exists():
    raise FileNotFoundError(
        "Could not find station_day.csv and stations.csv. "
        "Extract the Kaggle ZIP and update DATA_DIR."
    )


## 3. Load original files and inspect schema

In [ ]:
station_day_raw = pd.read_csv(STATION_DAY_FILE)
stations_raw = pd.read_csv(STATIONS_FILE)

print("station_day shape:", station_day_raw.shape)
print("stations shape:", stations_raw.shape)
print("\nstation_day columns:", station_day_raw.columns.tolist())
print("\nstations columns:", stations_raw.columns.tolist())

display(station_day_raw.head())
display(stations_raw.head())


## 4. Standardize column names and identify required fields

In [ ]:
def standardize_columns(frame):
    out = frame.copy()
    out.columns = (
        out.columns.astype(str).str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return out

station_day = standardize_columns(station_day_raw)
stations = standardize_columns(stations_raw)

def first_existing(frame, options):
    for col in options:
        if col in frame.columns:
            return col
    return None

station_id_day = first_existing(station_day, ["stationid", "station_id"])
station_id_meta = first_existing(stations, ["stationid", "station_id"])
date_col = first_existing(station_day, ["date", "last_update", "datetime", "timestamp"])
lat_col = first_existing(stations, ["latitude", "lat"])
lon_col = first_existing(stations, ["longitude", "lon", "lng"])
station_name_col = first_existing(stations, ["stationname", "station_name", "name"])
city_meta_col = first_existing(stations, ["city", "city_name"])
state_col = first_existing(stations, ["state"])

if not all([station_id_day, station_id_meta, date_col]):
    raise ValueError(
        "Required station ID/date columns not detected. "
        f"station_day columns={station_day.columns.tolist()}, "
        f"stations columns={stations.columns.tolist()}"
    )

print({
    "station_id_day": station_id_day,
    "station_id_meta": station_id_meta,
    "date": date_col,
    "latitude": lat_col,
    "longitude": lon_col,
    "station_name": station_name_col,
    "city": city_meta_col,
    "state": state_col
})


## 5. Join monitoring observations to station metadata

This is the integration step. It adds latitude, longitude and station labels to each daily station observation. The raw files remain unchanged.

In [ ]:
# Avoid duplicate join-key rows in metadata
stations_meta = stations.copy()
stations_meta = stations_meta.drop_duplicates(subset=[station_id_meta])

merged = station_day.merge(
    stations_meta,
    left_on=station_id_day,
    right_on=station_id_meta,
    how="left",
    suffixes=("", "_meta")
)

merged[date_col] = pd.to_datetime(merged[date_col], errors="coerce")

print("Joined shape:", merged.shape)
print("Rows without a matched station ID:", merged[station_id_meta].isna().sum())
display(merged.head())


## 6. Geographic filtering for Mumbai and Navi Mumbai

The filter first uses explicit station city/name text when available. If the metadata doesn't identify either area clearly, approximate latitude/longitude bounding boxes are used. Coordinate boxes are starting points, not official administrative boundaries; review the selected stations before final submission.

If station coordinates are absent or both cities cannot be separated reliably, do not fabricate records. Review the station metadata and adjust the boxes only after checking station locations.

In [ ]:
# Approximate configurable bounding boxes: (min_lat, max_lat, min_lon, max_lon)
# These are intentionally approximate and should be validated against station locations.
MUMBAI_BBOX = (18.85, 19.30, 72.75, 73.00)
NAVI_MUMBAI_BBOX = (18.95, 19.15, 73.00, 73.15)

def text_series(frame, col):
    if col and col in frame.columns:
        return frame[col].fillna("").astype(str).str.strip()
    return pd.Series("", index=frame.index, dtype="object")

if lat_col and lon_col:
    merged[lat_col] = pd.to_numeric(merged[lat_col], errors="coerce")
    merged[lon_col] = pd.to_numeric(merged[lon_col], errors="coerce")
else:
    raise ValueError("Latitude/longitude columns were not found in stations.csv.")

station_text = text_series(merged, station_name_col).str.lower()
city_text = text_series(merged, city_meta_col).str.lower()
state_text = text_series(merged, state_col).str.lower()

lat = merged[lat_col]
lon = merged[lon_col]

m_minlat, m_maxlat, m_minlon, m_maxlon = MUMBAI_BBOX
n_minlat, n_maxlat, n_minlon, n_maxlon = NAVI_MUMBAI_BBOX

in_mumbai_bbox = lat.between(m_minlat, m_maxlat) & lon.between(m_minlon, m_maxlon)
in_navi_bbox = lat.between(n_minlat, n_maxlat) & lon.between(n_minlon, n_maxlon)

# Explicit station/city text has priority. Coordinate classification is fallback.
explicit_navi = (
    city_text.str.contains(r"navi\s*mumbai", regex=True, na=False)
    | station_text.str.contains(r"navi\s*mumbai", regex=True, na=False)
)
explicit_mumbai = (
    city_text.str.fullmatch(r"mumbai", na=False)
    | station_text.str.contains(r"\bmumbai\b", regex=True, na=False)
) & ~explicit_navi

merged["project_city"] = np.select(
    [
        explicit_navi,
        explicit_mumbai,
        in_navi_bbox,
        in_mumbai_bbox
    ],
    [
        "Navi Mumbai",
        "Mumbai",
        "Navi Mumbai",
        "Mumbai"
    ],
    default="Outside Project Area"
)

# Keep only the two project areas
project = merged[merged["project_city"].isin(["Mumbai", "Navi Mumbai"])].copy()

print("Selected observations:", len(project))
print("\nRows by project city:")
display(project["project_city"].value_counts().rename_axis("City").to_frame("Rows"))

print("\nSelected station summary:")
station_summary_cols = ["project_city"]
if station_name_col:
    station_summary_cols.append(station_name_col)
station_summary_cols += [lat_col, lon_col]
selected_stations = project[station_summary_cols].drop_duplicates()
display(selected_stations.sort_values(["project_city"]))

print("\nCoordinate coverage:")
print(project[[lat_col, lon_col]].notna().all(axis=1).mean().round(3) * 100, "% rows have both coordinates")


In [ ]:
# Hard validation: both places must be represented, otherwise pause and inspect.
found_cities = set(project["project_city"].unique())
required_cities = {"Mumbai", "Navi Mumbai"}
missing_cities = required_cities - found_cities

if missing_cities:
    print("WARNING: Could not confidently select both cities:", sorted(missing_cities))
    print("Do not proceed to claim a two-city comparison yet.")
    print("Review the station summary and coordinate boxes above.")
    print("You may need an additional public source for Navi Mumbai if the 2015–2020 Kaggle station data has no suitable station.")
else:
    print("Both Mumbai and Navi Mumbai are represented in the filtered data.")


## 7. Save the geographically filtered source data

In [ ]:
filtered_file = OUTPUT_DIR / "mumbai_navi_mumbai_station_day_filtered_raw.csv"
project.to_csv(filtered_file, index=False)
print("Saved:", filtered_file.resolve())


## 8. Data quality audit and cleaning

- Convert pollutant and AQI fields to numeric.
- Remove exact duplicate observations.
- Remove rows without date or target AQI.
- Do not automatically delete legitimate high pollution values as outliers.
- Impute missing predictor values inside the model pipeline using training data only.

In [ ]:
# Numeric columns expected in station_day data (keep only those that exist)
target_col = first_existing(project, ["aqi"])
if target_col is None:
    raise ValueError("AQI target column not found in station_day.csv.")

# Identify numeric pollutant/environment columns from known pollutant names.
pollutant_candidates = [
    "pm2_5", "pm10", "no", "no2", "nox", "nh3", "so2", "co", "ozone", "o3",
    "benzene", "toluene", "xylene"
]
predictor_cols = [c for c in pollutant_candidates if c in project.columns]

if not predictor_cols:
    # Fallback: all numeric columns except IDs, coordinates, AQI and likely index fields.
    exclude = {target_col, lat_col, lon_col, station_id_day, station_id_meta}
    predictor_cols = [
        c for c in project.select_dtypes(include=np.number).columns
        if c not in exclude and not c.endswith("_id")
    ]

print("Target:", target_col)
print("Candidate pollutant predictors:", predictor_cols)

before_rows = len(project)
clean = project.copy()
clean = clean.drop_duplicates()

clean[date_col] = pd.to_datetime(clean[date_col], errors="coerce")
clean[target_col] = pd.to_numeric(clean[target_col], errors="coerce")

for col in predictor_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

clean = clean.dropna(subset=[date_col, target_col])
clean = clean[np.isfinite(clean[target_col])].copy()

# AQI should not be negative. Keep extreme positive values for inspection rather than deleting them.
clean = clean[clean[target_col] >= 0].copy()

print("Rows before cleaning:", before_rows)
print("Rows after cleaning:", len(clean))
print("Rows removed:", before_rows - len(clean))
print("\nMissing values in candidate fields:")
display(clean[[target_col] + predictor_cols].isna().sum().sort_values(ascending=False).to_frame("Missing"))


## 9. Feature engineering and AQI categories

In [ ]:
clean["date"] = clean[date_col].dt.normalize()
clean["year"] = clean[date_col].dt.year
clean["month"] = clean[date_col].dt.month
clean["month_name"] = clean[date_col].dt.month_name()
clean["quarter"] = clean[date_col].dt.quarter
clean["day_of_week"] = clean[date_col].dt.dayofweek
clean["day_name"] = clean[date_col].dt.day_name()
clean["weekend"] = (clean[date_col].dt.dayofweek >= 5).astype(int)

def season_from_month(month):
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Summer"
    if month in [6, 7, 8, 9]:
        return "Monsoon"
    if month in [10, 11]:
        return "Post-Monsoon"
    return np.nan

def aqi_category(value):
    if pd.isna(value): return np.nan
    if value <= 50: return "Good"
    if value <= 100: return "Satisfactory"
    if value <= 200: return "Moderate"
    if value <= 300: return "Poor"
    if value <= 400: return "Very Poor"
    return "Severe"

clean["season"] = clean["month"].apply(season_from_month)
clean["aqi_category"] = clean[target_col].apply(aqi_category)

cleaned_file = OUTPUT_DIR / "mumbai_navi_mumbai_aqi_cleaned.csv"
clean.to_csv(cleaned_file, index=False)
print("Saved cleaned dataset:", cleaned_file.resolve())
display(clean.head())


## 10. EDA: AQI by city, year and season

In [ ]:
display(
    clean.groupby("project_city")[target_col]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

plt.figure(figsize=(8, 5))
sns.boxplot(data=clean, x="project_city", y=target_col)
plt.title("Daily Station AQI Distribution: Mumbai vs Navi Mumbai")
plt.xlabel("City")
plt.ylabel("AQI")
plt.tight_layout()
plt.show()

yearly = clean.groupby(["year", "project_city"], as_index=False)[target_col].mean()
plt.figure(figsize=(10, 5))
sns.lineplot(data=yearly, x="year", y=target_col, hue="project_city", marker="o")
plt.title("Average AQI by Year and City")
plt.xlabel("Year")
plt.ylabel("Mean AQI")
plt.tight_layout()
plt.show()

seasonal = clean.groupby(["season", "project_city"], as_index=False)[target_col].mean()
plt.figure(figsize=(10, 5))
sns.barplot(data=seasonal, x="season", y=target_col, hue="project_city")
plt.title("Average AQI by Season and City")
plt.xlabel("Season")
plt.ylabel("Mean AQI")
plt.tight_layout()
plt.show()


## 11. Build a leakage-aware Random Forest regression model

This model estimates AQI from pollutant readings for the same observation. We intentionally exclude latitude/longitude from predictors: they are used to geographically filter the data and remain available for Power BI maps. The target AQI itself is never included as a feature.

The notebook uses a chronological 80/20 split. The imputer is fitted only on training data via an sklearn pipeline.

In [ ]:
# Keep only predictors that have at least some observed values and vary.
predictor_cols = [
    c for c in predictor_cols
    if c in clean.columns and clean[c].notna().any()
]

# Avoid target, IDs, coordinates, date/time-derived fields, and categorical labels as predictors.
blocked = {
    target_col, lat_col, lon_col, station_id_day, station_id_meta,
    "year", "month", "quarter", "day_of_week", "weekend"
}
predictor_cols = [c for c in predictor_cols if c not in blocked]

if len(predictor_cols) == 0:
    raise ValueError("No usable pollutant predictor columns found.")

model_data = clean.sort_values(date_col).reset_index(drop=True).copy()
split_at = int(len(model_data) * 0.80)

if split_at < 2 or len(model_data) - split_at < 1:
    raise ValueError("Not enough observations for an 80/20 train/test split.")

train_df = model_data.iloc[:split_at].copy()
test_df = model_data.iloc[split_at:].copy()

X_train = train_df[predictor_cols]
y_train = train_df[target_col]
X_test = test_df[predictor_cols]
y_test = test_df[target_col]

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("random_forest", RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training dates:", train_df[date_col].min(), "to", train_df[date_col].max())
print("Testing dates:", test_df[date_col].min(), "to", test_df[date_col].max())
print("Predictors:", predictor_cols)


## 12. Evaluate model: MAE, RMSE and R²

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred) if len(y_test) > 1 else np.nan

metrics = pd.DataFrame([
    {"Metric": "MAE", "Value": mae},
    {"Metric": "RMSE", "Value": rmse},
    {"Metric": "R2", "Value": r2},
    {"Metric": "Train_Rows", "Value": len(train_df)},
    {"Metric": "Test_Rows", "Value": len(test_df)}
])
display(metrics)

metrics_file = OUTPUT_DIR / "model_metrics.csv"
metrics.to_csv(metrics_file, index=False)
print("Saved:", metrics_file.resolve())


In [ ]:
# Actual vs predicted
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.55)
lo = min(float(y_test.min()), float(np.min(y_pred)))
hi = max(float(y_test.max()), float(np.max(y_pred)))
plt.plot([lo, hi], [lo, hi], linestyle="--")
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Actual vs Predicted AQI (Chronological Test Set)")
plt.tight_layout()
plt.show()

# Prediction error distribution: positive = model underpredicted AQI
errors = y_test.to_numpy() - y_pred
plt.figure(figsize=(9, 5))
sns.histplot(errors, bins=35, kde=True)
plt.axvline(0, linestyle="--")
plt.xlabel("Actual AQI - Predicted AQI")
plt.ylabel("Number of observations")
plt.title("Prediction Error Distribution")
plt.tight_layout()
plt.show()


## 13. Create test predictions and feature importance

In [ ]:
test_output = test_df.copy()
test_output["predicted_aqi"] = y_pred
test_output["prediction_error"] = test_output[target_col] - test_output["predicted_aqi"]
test_output["absolute_error"] = test_output["prediction_error"].abs()
test_output["residual"] = test_output["prediction_error"]
test_output["model_split"] = "Test"
test_output["predicted_aqi_category"] = test_output["predicted_aqi"].apply(aqi_category)

# Export feature importance from the fitted Random Forest.
rf = model.named_steps["random_forest"]
feature_importance = pd.DataFrame({
    "Feature": predictor_cols,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

display(feature_importance)

plt.figure(figsize=(9, 5))
sns.barplot(data=feature_importance, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

importance_file = OUTPUT_DIR / "feature_importance.csv"
feature_importance.to_csv(importance_file, index=False)
print("Saved:", importance_file.resolve())


## 14. Build the Power BI-ready dataset

The export keeps every cleaned observation. Predictions are populated only for the held-out test period; training rows have blank prediction fields. This avoids presenting in-sample predictions as independent test results.

`project_city`, `latitude`, and `longitude` support maps. `date`, `year`, `month`, `season`, and `aqi_category` support time-series and slicers.

In [ ]:
powerbi = model_data.copy()
powerbi["predicted_aqi"] = np.nan
powerbi["prediction_error"] = np.nan
powerbi["absolute_error"] = np.nan
powerbi["residual"] = np.nan
powerbi["predicted_aqi_category"] = np.nan
powerbi["model_split"] = "Train"

# Assign test output by the test rows' original model_data index.
test_indices = test_df.index
powerbi.loc[test_indices, "predicted_aqi"] = y_pred
powerbi.loc[test_indices, "prediction_error"] = test_df[target_col].to_numpy() - y_pred
powerbi.loc[test_indices, "absolute_error"] = np.abs(test_df[target_col].to_numpy() - y_pred)
powerbi.loc[test_indices, "residual"] = test_df[target_col].to_numpy() - y_pred
powerbi.loc[test_indices, "predicted_aqi_category"] = [aqi_category(v) for v in y_pred]
powerbi.loc[test_indices, "model_split"] = "Test"

# Friendly Power BI field names while retaining the source columns.
rename_map = {}
if target_col != "aqi":
    rename_map[target_col] = "aqi"
if lat_col and lat_col != "latitude":
    rename_map[lat_col] = "latitude"
if lon_col and lon_col != "longitude":
    rename_map[lon_col] = "longitude"
if station_name_col and station_name_col != "station_name":
    rename_map[station_name_col] = "station_name"

powerbi = powerbi.rename(columns=rename_map)

powerbi_file = OUTPUT_DIR / "mumbai_navi_mumbai_aqi_powerbi.csv"
powerbi.to_csv(powerbi_file, index=False)

print("Saved Power BI CSV:", powerbi_file.resolve())
print("Rows:", len(powerbi), "| Columns:", len(powerbi.columns))
display(powerbi.head())


## 15. City-wise test metrics and final validation

In [ ]:
city_metric_rows = []
for city, group in test_output.groupby("project_city"):
    if len(group) == 0:
        continue
    city_metric_rows.append({
        "City": city,
        "Test_Rows": len(group),
        "MAE": mean_absolute_error(group[target_col], group["predicted_aqi"]),
        "RMSE": np.sqrt(mean_squared_error(group[target_col], group["predicted_aqi"])),
        "R2": r2_score(group[target_col], group["predicted_aqi"]) if len(group) > 1 else np.nan
    })

city_metrics = pd.DataFrame(city_metric_rows)
display(city_metrics)

city_metrics_file = OUTPUT_DIR / "city_model_metrics.csv"
city_metrics.to_csv(city_metrics_file, index=False)

print("\nFinal city coverage:")
display(powerbi["project_city"].value_counts(dropna=False).to_frame("Rows"))

print("\nDate coverage:")
print(powerbi[date_col].min(), "to", powerbi[date_col].max())

print("\nOutput files:")
for path in [
    filtered_file, cleaned_file, powerbi_file,
    metrics_file, importance_file, city_metrics_file
]:
    print("-", path.resolve())


## 16. Notes for Power BI and viva

1. Import `mumbai_navi_mumbai_aqi_powerbi.csv` into Power BI.
2. Set `latitude` and `longitude` to **Decimal number** and assign their Data Categories as Latitude and Longitude.
3. Use `project_city` as the city slicer; use `date`, `year`, `month_name`, `season`, and `aqi_category` for time/category analysis.
4. Use `model_split = Test` when displaying Actual vs Predicted AQI and prediction errors. Training rows intentionally have blank predictions.
5. Use `model_metrics.csv` for MAE/RMSE/R² KPI cards, and `feature_importance.csv` for the model feature-importance bar chart.
6. Before final presentation, validate the selected station coordinates and the actual coverage years. Geographic bounding boxes are approximations.
7. If the classic Kaggle station data does not contain stations in Navi Mumbai, state that limitation and integrate a legitimate second source. Do not manufacture Navi Mumbai rows.

The rubric requires the original and preprocessed datasets, preprocessing documentation, ML notebook, interactive dashboard, and visualized model results. This notebook generates the data and metrics needed for those deliverables.